# Embeddings y Búsqueda Vectorial
## Motor de búsqueda semántica de ofertas laborales

**Proyecto Integrador 1 — Ingeniería de Sistemas**

### Objetivo del notebook

A partir del dataset ya limpio (generado en `02_Preprocesamiento.ipynb`), cubrir:

1. **Primera iteración de representación semántica** con Sentence-BERT (Sentence Transformers).
2. **Búsqueda vectorial baseline** con FAISS.
3. **Exploración de bases de datos vectoriales** (Pinecone, Chroma y alternativas) frente a FAISS.

Este notebook asume que ya ejecutaste `01_EDA.ipynb` y `02_Preprocesamiento.ipynb` al menos una vez (para que el parquet limpio exista en Google Drive).

## 0. Preparación e importaciones

Se instalan/importan las librerías necesarias y se monta Google Drive para acceder al dataset limpio guardado en el notebook anterior.

In [ ]:
# !pip install -q sentence-transformers faiss-cpu tqdm pinecone-client chromadb


In [ ]:
import os
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 1. Carga del dataset limpio

Se monta Google Drive (misma carpeta `proyecto_integrador` usada en el notebook de preprocesamiento) y se carga el parquet ya limpio, en vez de volver a descargar y limpiar el CSV crudo.

In [ ]:
from pathlib import Path

MONTAR_DRIVE = True  # cambia a False si prefieres trabajar solo con el almacenamiento efimero de Colab

if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/proyecto_integrador")
else:
    BASE_DIR = Path("/content")

RUTA_PROCESSED = BASE_DIR / "data" / "processed"
RUTA_LIMPIO = RUTA_PROCESSED / "job_descriptions_clean.parquet"

if not RUTA_LIMPIO.exists():
    raise FileNotFoundError(
        f"No se encontró {RUTA_LIMPIO}. "
        "Ejecuta primero 02_Preprocesamiento.ipynb para generarlo."
    )

df_clean = pd.read_parquet(RUTA_LIMPIO)
print(f"Registros cargados: {len(df_clean):,}")
df_clean[["Job Title", "texto_combinado"]].head(2)


## 2. Primera iteración de embeddings con Sentence-BERT

Al revisar una muestra de 50,000 registros se encontró que solo el **7.5%** de los textos combinados son únicos (3,760 de 50,000). Esto indica que el dataset es sintético: reutiliza un número relativamente pequeño de plantillas de texto (Job Title + Role + Qualifications + skills + Responsibilities + Job Description) y las replica con distintos países, empresas, salarios y modalidades.

Esto cambia la estrategia: en vez de generar un embedding por cada una de las 1,615,940 filas (carísimo y redundante, ya que muchas filas tendrían el **mismo vector exacto**), conviene:

1. **Deduplicar** por `texto_combinado` sobre **todo el dataset**, no solo una muestra.
2. Generar embeddings **solo de las plantillas únicas** (probablemente unos pocos miles).
3. Mantener un mapeo `template_id → todas las ofertas (Job Id, país, salario, modalidad, etc.) que comparten esa plantilla`.
4. Al buscar: encontrar la(s) plantilla(s) más similares con FAISS, y luego expandir a las ofertas reales que la comparten, aplicando ahí los filtros estructurados (país, modalidad, salario, experiencia).

Esto permite trabajar con el **dataset completo** (no una muestra) sin disparar el costo computacional, porque el costo de embeddings depende del número de plantillas únicas, no del número de filas.

### 2.1 Diagnóstico de duplicación en el dataset completo

Antes de decidir cuántas plantillas hay que embeber, se confirma la magnitud real sobre las 1,615,940 filas (no solo la muestra de 50,000 usada antes).

In [ ]:
n_total = len(df_clean)
n_unicos = df_clean["texto_combinado"].nunique()

print(f"Ofertas totales: {n_total:,}")
print(f"Textos combinados únicos: {n_unicos:,} ({n_unicos / n_total:.2%})")


### 2.2 Construcción de la tabla de plantillas únicas

Se asigna un `template_id` a cada texto combinado distinto, y se construye una tabla `plantillas` con una fila por plantilla única — esa es la que se va a embeber. El `template_id` se agrega también a `df_clean` para poder expandir después los resultados de la búsqueda hacia todas las ofertas reales.

In [ ]:
df_clean["template_id"] = df_clean.groupby("texto_combinado", sort=False).ngroup()

plantillas = (
    df_clean
    .drop_duplicates(subset="template_id")
    .loc[:, ["template_id", "texto_combinado", "Job Title", "Role"]]
    .sort_values("template_id")
    .reset_index(drop=True)
)

print(f"Plantillas únicas a embeber: {len(plantillas):,}")
plantillas.head()


### 2.3 Generación de embeddings (sobre las plantillas, no sobre las 1.6M filas)

Se usa `all-MiniLM-L6-v2`: modelo pequeño (384 dimensiones), rápido, buen punto de partida para *semantic search*. Si el desempeño no es suficiente en la fase de evaluación, se puede migrar a `all-mpnet-base-v2` (mejor calidad, más lento) en una segunda iteración — al ser pocas plantillas, cambiar de modelo es barato de volver a correr.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "all-MiniLM-L6-v2"

modelo = SentenceTransformer(MODELO_EMBEDDINGS)
print(f"Modelo cargado: {MODELO_EMBEDDINGS}")
print(f"Dimensión del embedding: {modelo.get_sentence_embedding_dimension()}")


In [ ]:
inicio = time.time()

embeddings = modelo.encode(
    plantillas["texto_combinado"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # normaliza a norma 1 -> producto punto == similitud coseno
)

duracion = time.time() - inicio
print(f"Embeddings generados: {embeddings.shape}")
print(f"Tiempo total: {duracion:.1f} s  ({duracion / len(plantillas) * 1000:.2f} ms/plantilla)")


In [ ]:
np.save(RUTA_PROCESSED / "embeddings_plantillas.npy", embeddings)
plantillas.to_parquet(RUTA_PROCESSED / "plantillas_meta.parquet", index=False)
df_clean[["Job Id", "template_id"]].to_parquet(RUTA_PROCESSED / "job_id_template_map.parquet", index=False)

print("Embeddings de plantillas y mapeo Job Id -> template_id guardados en Drive.")


### 2.4 Búsqueda con FAISS (baseline)

Antes de explorar bases de datos vectoriales externas, se valida el pipeline con **FAISS** (tal como se planteó en el anteproyecto), usando un índice plano (`IndexFlatIP`) sobre las plantillas únicas.

In [ ]:
import faiss

dimension = embeddings.shape[1]
indice_faiss = faiss.IndexFlatIP(dimension)  # producto interno == coseno, porque los vectores están normalizados
indice_faiss.add(embeddings)

print(f"Plantillas indexadas en FAISS: {indice_faiss.ntotal:,}")


In [ ]:
def buscar_ofertas(consulta: str, k_plantillas: int = 5, max_resultados: int = 20, filtros: dict | None = None):
    """
    Busca ofertas semánticamente similares a `consulta`.

    - k_plantillas: cuántas plantillas de texto distintas considerar como relevantes.
    - max_resultados: máximo de ofertas individuales a devolver una vez expandidas las plantillas.
    - filtros: dict opcional de {columna: valor} sobre columnas de df_clean, p. ej. {"Country": "Colombia"}.
    """
    vector_consulta = modelo.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    similitudes, indices = indice_faiss.search(vector_consulta, k_plantillas)

    template_ids_relevantes = plantillas.iloc[indices[0]]["template_id"].values
    similitud_por_template = dict(zip(template_ids_relevantes, similitudes[0]))

    resultados = df_clean[df_clean["template_id"].isin(template_ids_relevantes)].copy()
    resultados["similitud"] = resultados["template_id"].map(similitud_por_template)

    if filtros:
        for columna, valor in filtros.items():
            resultados = resultados[resultados[columna] == valor]

    resultados = resultados.sort_values("similitud", ascending=False)
    return resultados[["Job Title", "Role", "Country", "Work Type", "Salary Range", "similitud"]].head(max_resultados)

# Ejemplo sin filtros
buscar_ofertas("python developer with machine learning and NLP experience", k_plantillas=5)


In [ ]:
# Ejemplo aplicando un filtro estructurado, como se plantea en el Objetivo específico 4
buscar_ofertas(
    "python developer with machine learning and NLP experience",
    k_plantillas=5,
    filtros={"Work Type": "Full-Time"},
)


## 3. Exploración de bases de datos vectoriales

El anteproyecto planteó **FAISS** como motor de búsqueda vectorial. FAISS es una biblioteca (no una base de datos): es muy rápida y gratuita, pero corre en memoria/proceso local, no maneja persistencia ni actualizaciones incrementales de forma nativa, y no está pensada como servicio con API propia. Para la etapa de integración (API + interfaz web) conviene comparar alternativas de **bases de datos vectoriales** administradas o auto-hospedadas.

| Opción | Tipo | Costo | Persistencia / escalabilidad | Filtrado por metadatos | Curva de aprendizaje | Cuándo conviene |
|---|---|---|---|---|---|---|
| **FAISS** | Biblioteca en memoria | Gratis | Manual (hay que guardar/cargar el índice); escala vertical | Limitado, se maneja aparte en Pandas/SQL | Baja | Prototipos, datasets que caben en memoria/disco de un solo servidor |
| **Chroma** | Base de datos vectorial embebida/self-hosted | Gratis (open source) | Persistencia en disco nativa; fácil de correr local o en Docker | Sí, nativo | Baja | Buen punto medio para pasar de FAISS a algo con persistencia sin salir de Python |
| **Qdrant** | Base de datos vectorial self-hosted o cloud (tiene *free tier*) | Gratis (self-host) / plan gratuito cloud limitado | Persistencia, filtrado avanzado, buena para producción | Sí, muy completo | Media | Cuando se quiere una API HTTP/gRPC propia y filtros complejos sobre metadatos |
| **Pinecone** | Base de datos vectorial 100% administrada (SaaS) | Plan gratuito limitado (una región, límite de vectores); pago por uso a partir de cierto volumen | Totalmente administrada, escala automáticamente | Sí | Baja (API simple) | Cuando no se quiere administrar infraestructura y se prioriza velocidad de desarrollo |
| **Weaviate** | Base de datos vectorial self-hosted o cloud | Gratis (self-host) / cloud con costo | Persistencia, soporta búsqueda híbrida (vectorial + palabras clave) | Sí | Media-alta | Si además de similitud semántica se quiere combinar con búsqueda léxica (BM25) |

**Recomendación para esta iteración:** dado que el equipo ya validó FAISS en el anteproyecto y el presupuesto no contempla servicios pagos recurrentes más allá de lo presupuestado, una ruta razonable es:

1. Mantener **FAISS** como baseline de referencia (ya funciona, es gratis, cero dependencias externas).
2. Probar **Pinecone** en su capa gratuita para evaluar si la facilidad de integración (API lista, sin mantener infraestructura) compensa el límite de vectores del plan free — es útil para la demo/API del alcance del proyecto.
3. Si el plan gratuito de Pinecone resulta insuficiente para 1.6M de registros, evaluar **Qdrant** o **Chroma** self-hosted en el mismo VPS ya presupuestado (Railway), que no tienen límite artificial de vectores.

A continuación, un ejemplo mínimo (no ejecutado, requiere API key) de cómo se vería la integración con Pinecone para comparar el mismo flujo de búsqueda.

In [ ]:
# Ejemplo ilustrativo de integración con Pinecone (requiere cuenta y API key gratuitas).
# No se ejecuta aquí porque necesita credenciales.
# Nota: se indexan las PLANTILLAS (texto único), no cada oferta individual, por la misma razón que con FAISS.
#
# from pinecone import Pinecone, ServerlessSpec
#
# pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
#
# NOMBRE_INDICE = "ofertas-laborales"
# if NOMBRE_INDICE not in [i.name for i in pc.list_indexes()]:
#     pc.create_index(
#         name=NOMBRE_INDICE,
#         dimension=dimension,          # 384 para all-MiniLM-L6-v2
#         metric="cosine",
#         spec=ServerlessSpec(cloud="aws", region="us-east-1"),
#     )
#
# indice_pinecone = pc.Index(NOMBRE_INDICE)
#
# vectores = [
#     (str(row["template_id"]), emb.tolist(), {"Job Title": row["Job Title"], "Role": row["Role"]})
#     for row, emb in zip(plantillas.to_dict("records"), embeddings)
# ]
# for lote in range(0, len(vectores), 100):
#     indice_pinecone.upsert(vectores[lote:lote + 100])
#
# resultado = indice_pinecone.query(
#     vector=modelo.encode("python developer with machine learning").tolist(),
#     top_k=5,
#     include_metadata=True,
# )


In [ ]:
# Ejemplo ilustrativo de integración con Chroma (sí se puede instalar localmente y probar sin API key externa).
# Igual que con Pinecone: se indexan las plantillas, no cada oferta individual.
# import chromadb
#
# cliente_chroma = chromadb.PersistentClient(path=str(RUTA_PROCESSED / "chroma_db"))
# coleccion = cliente_chroma.get_or_create_collection(name="ofertas_laborales", metadata={"hnsw:space": "cosine"})
#
# coleccion.add(
#     ids=[str(t) for t in plantillas["template_id"]],
#     embeddings=embeddings.tolist(),
#     metadatas=plantillas[["Job Title", "Role"]].to_dict("records"),
# )
#
# resultados_chroma = coleccion.query(
#     query_embeddings=modelo.encode(["python developer with machine learning"]).tolist(),
#     n_results=5,
# )


## 4. Comparación cuantitativa: `all-MiniLM-L6-v2` vs `all-mpnet-base-v2`

Antes de decidir si vale la pena migrar a un modelo más grande, se comparan ambos sobre el mismo conjunto de plantillas y las mismas consultas de prueba, midiendo:

1. **Tiempo de generación de embeddings** sobre las 3,760 plantillas (costo de indexar).
2. **Tiempo de codificación por consulta** (costo de cada búsqueda en producción).
3. **Grado de acuerdo entre modelos**: para cada consulta, qué tanto se solapan las top-k plantillas que devuelve cada uno (si ambos coinciden casi siempre, el modelo pequeño ya es suficiente; si difieren mucho, vale la pena mirar con más cuidado cuál da resultados más relevantes).
4. **Inspección cualitativa** de los resultados lado a lado, para juzgar a ojo cuál interpreta mejor la intención de la consulta.

`all-mpnet-base-v2` es el modelo de calidad más alta recomendado por Sentence-Transformers para *semantic search* en inglés (768 dimensiones, 12 capas), a costa de ser más lento que MiniLM.

### 4.1 Cargar el segundo modelo y generar sus embeddings

Se reutiliza la misma tabla `plantillas` (3,760 textos únicos) y el mismo enfoque de normalización, para que la comparación sea justa.

In [ ]:
MODELO_EMBEDDINGS_2 = "all-mpnet-base-v2"

modelo_2 = SentenceTransformer(MODELO_EMBEDDINGS_2)
print(f"Modelo cargado: {MODELO_EMBEDDINGS_2}")
print(f"Dimensión del embedding: {modelo_2.get_sentence_embedding_dimension()}")

inicio = time.time()
embeddings_2 = modelo_2.encode(
    plantillas["texto_combinado"].tolist(),
    batch_size=64,  # lotes mas chicos porque el modelo es mas pesado
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
duracion_2 = time.time() - inicio

print(f"Embeddings generados: {embeddings_2.shape}")
print(f"Tiempo total: {duracion_2:.1f} s  ({duracion_2 / len(plantillas) * 1000:.2f} ms/plantilla)")
print(f"Comparación: {MODELO_EMBEDDINGS} tardó {duracion:.1f} s en el mismo paso (celda 2.3).")


### 4.2 Índice FAISS para el segundo modelo

In [ ]:
indice_faiss_2 = faiss.IndexFlatIP(embeddings_2.shape[1])
indice_faiss_2.add(embeddings_2)

print(f"Plantillas indexadas (modelo 2): {indice_faiss_2.ntotal:,}")


### 4.3 Consultas de prueba

Un pequeño set de consultas representativas del tipo de búsqueda que haría un usuario real: mezcla de perfiles técnicos, no técnicos, y frases cortas vs. descriptivas.

In [ ]:
CONSULTAS_PRUEBA = [
    "python developer with machine learning and NLP experience",
    "senior backend engineer with cloud and microservices experience",
    "marketing manager with social media and branding skills",
    "entry level data entry no experience required",
    "financial analyst with excel and forecasting skills",
]


### 4.4 Función de comparación por consulta

Para cada consulta, se corre la búsqueda con ambos modelos/índices, se mide el tiempo de codificación de cada uno, y se calcula el **overlap** (cuántas de las top-k plantillas coinciden entre los dos modelos, sin importar el orden).

In [ ]:
def comparar_modelos(consulta: str, k: int = 5):
    # Modelo 1: all-MiniLM-L6-v2
    inicio_1 = time.time()
    vector_1 = modelo.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    _, indices_1 = indice_faiss.search(vector_1, k)
    tiempo_1 = time.time() - inicio_1
    top_1 = plantillas.iloc[indices_1[0]]
    templates_1 = set(top_1["template_id"].values)
    roles_1 = set(top_1["Role"].values)

    # Modelo 2: all-mpnet-base-v2
    inicio_2 = time.time()
    vector_2 = modelo_2.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    _, indices_2 = indice_faiss_2.search(vector_2, k)
    tiempo_2 = time.time() - inicio_2
    top_2 = plantillas.iloc[indices_2[0]]
    templates_2 = set(top_2["template_id"].values)
    roles_2 = set(top_2["Role"].values)

    overlap_estricto = len(templates_1 & templates_2)   # misma plantilla exacta
    overlap_por_rol = len(roles_1 & roles_2)             # misma familia de "Role", plantilla puede diferir

    comparacion = pd.DataFrame({
        "template_id MiniLM": top_1["template_id"].values,
        f"Top-{k} MiniLM (Job Title / Role)": (top_1["Job Title"] + " / " + top_1["Role"]).values,
        "template_id MPNet": top_2["template_id"].values,
        f"Top-{k} MPNet (Job Title / Role)": (top_2["Job Title"] + " / " + top_2["Role"]).values,
    })

    print(f"Consulta: \"{consulta}\"")
    print(f"  Tiempo MiniLM: {tiempo_1 * 1000:.1f} ms | Tiempo MPNet: {tiempo_2 * 1000:.1f} ms")
    print(f"  Coincidencia por plantilla exacta: {overlap_estricto}/{k}")
    print(f"  Coincidencia por familia de Role:  {overlap_por_rol}/{min(len(roles_1), len(roles_2))} roles distintos en comun")
    return comparacion


### 4.5 Resultados sobre todas las consultas de prueba

In [ ]:
resumen_overlap = []

for consulta in CONSULTAS_PRUEBA:
    tabla = comparar_modelos(consulta, k=5)
    display(tabla)
    print()


### 4.6 Cómo interpretar esta comparación

- **Si el overlap es alto (4-5 de 5) en la mayoría de consultas**: los dos modelos están de acuerdo en qué plantillas son relevantes. En ese caso, `all-MiniLM-L6-v2` ya captura bien la semántica necesaria para este dataset, y no se justifica pagar el costo extra de velocidad de `all-mpnet-base-v2` — sobre todo considerando que el dataset solo tiene 3,760 plantillas realmente distintas (poca variedad semántica que discriminar).
- **Si el overlap es bajo en varias consultas**: revisar manualmente cuál de los dos conjuntos de resultados luce más relacionado con la consulta (por ejemplo, ¿"marketing manager" realmente trae perfiles de marketing, o se cuela algo de ventas/diseño?). Si `all-mpnet-base-v2` da resultados visiblemente más precisos, vale la pena adoptarlo para la fase de evaluación formal (Objetivo específico 5), aceptando el costo de ser ~2-3x más lento por consulta.
- **La diferencia de tiempos** (impresa en cada consulta) indica el costo real de cambiar de modelo en producción — recuerda que ese costo se paga en cada consulta de un usuario, no solo una vez como con los embeddings de las plantillas.

Con solo 3,760 plantillas, ambos modelos tardan segundos en total, así que la decisión aquí es principalmente de **calidad de resultados**, no de tiempo de cómputo.

## 5. Próximos pasos

1. **Decidir la base de datos vectorial** definitiva a partir de esta comparación (recomendado: probar Pinecone free tier + Chroma self-hosted, y quedarse con el que mejor equilibre costo/latencia/facilidad de integración con la API).
2. **Escalar el embedding** al dataset completo (o a un subconjunto representativo mayor) una vez elegida la base de datos, idealmente en un proceso por lotes fuera del notebook.
3. ~~Evaluar si `all-MiniLM-L6-v2` es suficiente o si conviene migrar a `all-mpnet-base-v2`~~ — ya resuelto en la sección 4: decidir con base en el overlap y la inspección cualitativa obtenidos ahí.
4. Con la base vectorial elegida, comenzar el **Objetivo específico 4**: desarrollo de la API y la interfaz web.